In [6]:
import json
from tqdm.notebook import tqdm

def load_all_instances(file_path: str):
    """Load all instances from an output.jsonl file."""
    # Count total lines first for accurate progress bar
    with open(file_path, 'r', encoding='utf-8') as f:
        total_lines = sum(1 for line in f if line.strip())

    instances = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in tqdm(f, total=total_lines, desc='Loading instances', unit='instance'):
            line = line.strip()
            if not line:
                continue
            raw = json.loads(line)
            vlog = raw.get('test_result', {}).get('validation_log', [])
            validation_map = {}
            for entry in vlog:
                idx = entry.get('step_index')
                if idx is not None:
                    validation_map[idx] = entry
            instances.append({
                'instance_id': raw.get('instance_id', ''),
                'instance': raw.get('instance', {}),
                'history': raw.get('history', []),
                'git_patch': raw.get('test_result', {}).get('git_patch', ''),
                'validation_log': vlog,
                'validation_map': validation_map,
                'metadata': raw.get('metadata', {}),
                'instruction': raw.get('instruction', ''),
                'metrics': raw.get('metrics', {}),
                'error': raw.get('error'),
            })
    return instances


# Example usage
output_jsonl = '/home/chentianyu/murong/code/OpenHands_SWE-Bench-Optimized/evaluation/evaluation_outputs/outputs/SWE-Gym__SWE-Gym-train/CodeActAgent/Qwen3-Coder-480B-A35B-Instruct_maxiter_100_N_v0.61.0-no-hint-train-qwen3_coder_480b_a35b_instruct-t05/Qwen3-Coder-480B-A35B-Instruct_maxiter_100_N_v0.61.0-no-hint-train-qwen3_coder_480b_a35b_instruct-t05-run_1/output.jsonl'
instances = load_all_instances(output_jsonl)
print(f'Total instances: {len(instances)}')


Loading instances:   0%|          | 0/2119 [00:00<?, ?instance/s]

Total instances: 2119


In [32]:
instances[0].keys()

dict_keys(['instance_id', 'instance', 'history', 'git_patch', 'validation_log', 'validation_map', 'metadata', 'instruction', 'metrics', 'error'])

In [7]:
instances[1]['instance_id']

'getmoto__moto-5876'

In [47]:
instances[0]['history'][235:240]

[{'id': 238,
  'timestamp': '2026-03-13T17:37:47.898982',
  'source': 'agent',
  'message': 'Command `cd /workspace/getmoto__moto__4.1 && git diff moto/dynamodb/responses.py` executed with exit code 0.',
  'cause': 237,
  'observation': 'run',
  'tool_call_metadata': {'function_name': 'execute_bash',
   'tool_call_id': 'call_76c0355eef5b4ed284192ff2',
   'model_response': {'id': '143edbce59b94e9daca76090e31c3c5d',
    'created': 1773423467,
    'model': 'zai-org/GLM-5-FP8',
    'object': 'chat.completion',
    'system_fingerprint': None,
    'choices': [{'finish_reason': 'tool_calls',
      'index': 0,
      'message': {'content': 'Perfect! Let me create a final summary and verify all changes are correct:',
       'role': 'assistant',
       'tool_calls': [{'index': 0,
         'function': {'arguments': '{"command": "cd /workspace/getmoto__moto__4.1 && git diff moto/dynamodb/responses.py", "security_risk": "LOW"}',
          'name': 'execute_bash'},
         'id': 'call_76c0355eef5b4ed

In [9]:
import re

def _normalize_text(x):
    if x is None:
        return ''
    return x if isinstance(x, str) else str(x)

def _is_embedded_tool_call_message(text: str) -> bool:
    """Detect non-native tool-call markup embedded in assistant message text."""
    if not text:
        return False
    return bool(
        re.search(r"<tool_call>|<function=|<parameter(?:=[^>]+)?>", text)
    )

def _extract_leading_narrative(text: str) -> str:
    """Keep any natural-language preface that appears before embedded tool markup."""
    if not text:
        return ''
    m = re.search(r"<tool_call>|<function=", text)
    if not m:
        return ''
    return text[:m.start()].strip()

def _parse_embedded_tool_call(text: str) -> dict:
    """Parse embedded tool-call markup into a compact summary.

    Also preserves leading narrative/thinking text when present, e.g.:
    "Let me inspect... <tool_call>execute_bash> ..."
    """
    tool_name = None
    narrative = _extract_leading_narrative(text)

    # Support formats like:
    # 1) <tool_call>execute_bash> ...
    # 2) <tool_call>function=execute_bash> ...
    # 3) <function=execute_bash> ... </function>
    m = re.search(r"<tool_call>\s*(?:function=)?([a-zA-Z0-9_:\-\.]+)\s*>", text)
    if not m:
        m = re.search(r"<function=([a-zA-Z0-9_:\-\.]+)>", text)
    if m:
        tool_name = m.group(1).strip()

    params = {}
    for pm in re.finditer(r"<parameter(?:=([^>]+))?>\s*(.*?)\s*</parameter>", text, re.DOTALL):
        key = (pm.group(1) or 'value').strip()
        value = pm.group(2).strip()
        params[key] = value

    # Build readable summary with stable key ordering for common params
    ordered_keys = ['command', 'path', 'view_range', 'security_risk', 'message', 'content', 'thought']
    lines = []
    for k in ordered_keys:
        if k in params and params[k]:
            lines.append(f'{k}: {params[k]}')
    for k, v in params.items():
        if k not in ordered_keys and v:
            lines.append(f'{k}: {v}')

    if not tool_name and lines:
        # Fallback for malformed snippets: infer from command-like content
        tool_name = 'tool_call'

    if lines:
        tool_content = f'[{tool_name}] {lines[0]}' if tool_name else lines[0]
        if len(lines) > 1:
            tool_content += '\n' + '\n'.join(lines[1:])
    else:
        tool_content = f'[{tool_name}]' if tool_name else text

    # Preserve natural-language thinking/planning text alongside parsed tool content
    if narrative:
        content = f'{narrative}\n\n{tool_content}'
    else:
        content = tool_content

    return {
        'tool': tool_name or 'tool_call',
        'content': content,
    }

def _is_tool_result_message(text: str) -> bool:
    """Detect plain-text tool results often seen in non-native histories."""
    t = (text or '').strip()
    if not t:
        return False
    return t.startswith('EXECUTION RESULT of [') or t.startswith('Retrieving content for:')

def _extract_native_reasoning(entry: dict) -> str:
    """Extract reasoning/plan text stored in tool_call_metadata.model_response."""
    meta = entry.get('tool_call_metadata') or {}
    model_resp = meta.get('model_response') or {}
    choices = model_resp.get('choices') or []
    if not choices:
        return ''

    msg = (choices[0] or {}).get('message') or {}
    reasoning = _normalize_text(msg.get('reasoning_content', '')).strip()
    assistant_content = _normalize_text(msg.get('content', '')).strip()

    parts = []
    if reasoning:
        parts.append(reasoning)
    if assistant_content and assistant_content not in reasoning:
        parts.append(assistant_content)
    return '\n\n'.join(parts).strip()

def _merge_reasoning(content: str, reasoning: str) -> str:
    """Attach reasoning before tool action content when present."""
    content = _normalize_text(content).strip()
    reasoning = _normalize_text(reasoning).strip()
    if not reasoning:
        return content
    if not content:
        return reasoning
    if reasoning in content:
        return content
    return f'[reasoning]\n{reasoning}\n\n[action]\n{content}'

def format_history_for_llm(history, max_message_len=1000):
    """Format an instance's history into a human/LLM-readable string.

    Auto-detects two history formats:
    1) Native tool parser format (structured action/args fields).
    2) Non-native format where tool calls are embedded in message text
       (e.g. <tool_call>...<parameter=...>...</parameter>).

    Actions (→) are things the agent/user actively does.
    Observations (←) are responses/results from the environment.
    """
    lines = []
    for i, entry in enumerate(history):
        source = entry.get('source', '?')
        action = entry.get('action', '')
        observation = entry.get('observation', '')
        message = _normalize_text(entry.get('message', ''))
        args = entry.get('args', {}) or {}
        native_reasoning = _extract_native_reasoning(entry)

        # Role label
        if source == 'agent':
            role = 'AGENT'
        elif source == 'user':
            role = 'USER'
        elif source == 'environment':
            role = 'ENV'
        else:
            role = _normalize_text(source).upper()

        # First, detect non-native embedded tool-call format in message text
        if _is_embedded_tool_call_message(message):
            parsed = _parse_embedded_tool_call(message)
            direction = '→'
            kind = f"action:{parsed['tool']}"
            content = parsed['content']

        # Then detect plain-text tool results from non-native format
        elif _is_tool_result_message(message):
            direction = '←'
            kind = 'obs:tool_result'
            content = message

        else:
            # Native structured format behavior (existing logic)
            if action:
                direction = '→'
                kind = f'action:{action}'
            elif observation:
                direction = '←'
                kind = f'obs:{observation}'
            else:
                direction = ' '
                kind = 'message'

            # Pick the most informative content per action/observation type
            if action == 'message':
                content = _normalize_text(args.get('content', '')) or message
            elif action == 'think':
                content = _normalize_text(args.get('thought', '')) or message or native_reasoning
            elif action == 'run':
                content = _normalize_text(args.get('command', '')) or message
            elif action in ('read', 'write', 'edit'):
                # args.command can be missing in native traces; fall back to action.
                cmd = _normalize_text(args.get('command', ''))
                label = cmd or action
                path = _normalize_text(args.get('path', ''))
                old_str = _normalize_text(args.get('old_str', ''))
                new_str = _normalize_text(args.get('new_str', ''))
                if cmd == 'str_replace' and (old_str or new_str):
                    content = f'[{label}] {path}\n--- old ---\n{old_str}\n+++ new +++\n{new_str}'
                elif cmd in ('view', 'create', 'insert', 'undo_edit'):
                    body = _normalize_text(args.get('file_text', args.get('new_str', message)))
                    content = f'[{label}] {path}' + (f'\n{body}' if body else '')
                else:
                    content = f'[{label}] {path}' if (label or path) else message
            elif observation:
                content = _normalize_text(entry.get('content', '')) or message
            else:
                content = _normalize_text(entry.get('content', '')) or _normalize_text(args.get('content', '')) or message

            # For native traces, many tool actions keep reasoning in model_response.
            if role == 'AGENT' and action in ('run', 'read', 'write', 'edit', 'browse', 'search', 'message'):
                content = _merge_reasoning(content, native_reasoning)

        # Truncate long content for readability
        # if content and len(content) > max_message_len and role in ('ENV', 'AGENT'):
        #     content = content[:max_message_len] + f'... [truncated, total {len(content)} chars]'

        lines.append(f'[{i:3d}] {role:<5} {direction} {kind}: {content}')

    return '\n\n'.join(lines)


print(format_history_for_llm(instances[1]['history']))

[  0] AGENT → action:system: You are OpenHands agent, a helpful AI assistant that can interact with a computer to solve tasks.

<ROLE>
Your primary role is to assist users by executing commands, modifying code, and solving technical problems effectively. You should be thorough, methodical, and prioritize quality over speed.
* If the user asks a question, like "why is X happening", don't try to fix the problem. Just give an answer to the question.
</ROLE>

<EFFICIENCY>
* Each action you take is somewhat expensive. Wherever possible, combine multiple actions into a single action, e.g. combine multiple bash commands into one, using sed and grep to edit/view multiple files at once.
* When exploring the codebase, use efficient tools like find, grep, and git commands with appropriate filters to minimize unnecessary operations.
</EFFICIENCY>

<FILE_SYSTEM_GUIDELINES>
* When a user provides a file path, do NOT assume it's relative to the current working directory. First explore the file system

In [50]:
import re

def _normalize_text(x):
    if x is None:
        return ''
    return x if isinstance(x, str) else str(x)

def _is_embedded_tool_call_message(text: str) -> bool:
    """Detect non-native tool-call markup embedded in assistant message text."""
    if not text:
        return False
    return bool(
        re.search(r"<tool_call>|<function=|<parameter(?:=[^>]+)?>", text)
    )

def _extract_leading_narrative(text: str) -> str:
    """Keep any natural-language preface that appears before embedded tool markup."""
    if not text:
        return ''
    m = re.search(r"<tool_call>|<function=", text)
    if not m:
        return ''
    return text[:m.start()].strip()

def _parse_embedded_tool_call(text: str) -> dict:
    """Parse embedded tool-call markup into a compact summary.

    Also preserves leading narrative/thinking text when present, e.g.:
    "Let me inspect... <tool_call>execute_bash> ..."
    """
    tool_name = None
    narrative = _extract_leading_narrative(text)

    # Support formats like:
    # 1) <tool_call>execute_bash> ...
    # 2) <tool_call>function=execute_bash> ...
    # 3) <function=execute_bash> ... </function>
    m = re.search(r"<tool_call>\s*(?:function=)?([a-zA-Z0-9_:\-\.]+)\s*>", text)
    if not m:
        m = re.search(r"<function=([a-zA-Z0-9_:\-\.]+)>", text)
    if m:
        tool_name = m.group(1).strip()

    params = {}
    for pm in re.finditer(r"<parameter(?:=([^>]+))?>\s*(.*?)\s*</parameter>", text, re.DOTALL):
        key = (pm.group(1) or 'value').strip()
        value = pm.group(2).strip()
        params[key] = value

    # Build readable summary with stable key ordering for common params
    ordered_keys = ['command', 'path', 'view_range', 'security_risk', 'message', 'content', 'thought']
    lines = []
    for k in ordered_keys:
        if k in params and params[k]:
            lines.append(f'{k}: {params[k]}')
    for k, v in params.items():
        if k not in ordered_keys and v:
            lines.append(f'{k}: {v}')

    if not tool_name and lines:
        # Fallback for malformed snippets: infer from command-like content
        tool_name = 'tool_call'

    if lines:
        tool_content = f'[{tool_name}] {lines[0]}' if tool_name else lines[0]
        if len(lines) > 1:
            tool_content += '\n' + '\n'.join(lines[1:])
    else:
        tool_content = f'[{tool_name}]' if tool_name else text

    # Preserve natural-language thinking/planning text alongside parsed tool content
    if narrative:
        content = f'{narrative}\n\n{tool_content}'
    else:
        content = tool_content

    return {
        'tool': tool_name or 'tool_call',
        'content': content,
    }

def _is_tool_result_message(text: str) -> bool:
    """Detect plain-text tool results often seen in non-native histories."""
    t = (text or '').strip()
    if not t:
        return False
    return t.startswith('EXECUTION RESULT of [') or t.startswith('Retrieving content for:')

def _extract_native_reasoning(entry: dict) -> str:
    """Extract reasoning/plan text stored in tool_call_metadata.model_response."""
    meta = entry.get('tool_call_metadata') or {}
    model_resp = meta.get('model_response') or {}
    choices = model_resp.get('choices') or []
    if not choices:
        return ''

    msg = (choices[0] or {}).get('message') or {}
    reasoning = _normalize_text(msg.get('reasoning_content', '')).strip()
    assistant_content = _normalize_text(msg.get('content', '')).strip()

    parts = []
    if reasoning:
        parts.append(reasoning)
    if assistant_content and assistant_content not in reasoning:
        parts.append(assistant_content)
    return '\n\n'.join(parts).strip()

def _merge_reasoning(content: str, reasoning: str) -> str:
    """Attach reasoning before tool action content when present."""
    content = _normalize_text(content).strip()
    reasoning = _normalize_text(reasoning).strip()
    if not reasoning:
        return content
    if not content:
        return reasoning
    if reasoning in content:
        return content
    return f'[reasoning]\n{reasoning}\n\n[action]\n{content}'

def format_history_for_llm(history, max_message_len=1000):
    """Format an instance's history into a human/LLM-readable string.

    Auto-detects two history formats:
    1) Native tool parser format (structured action/args fields).
    2) Non-native format where tool calls are embedded in message text
       (e.g. <tool_call>...<parameter=...>...</parameter>).

    Actions (→) are things the agent/user actively does.
    Observations (←) are responses/results from the environment.
    """
    lines = []
    for i, entry in enumerate(history):
        source = entry.get('source', '?')
        action = entry.get('action', '')
        observation = entry.get('observation', '')
        message = _normalize_text(entry.get('message', ''))
        args = entry.get('args', {}) or {}
        native_reasoning = _extract_native_reasoning(entry)

        # Role label
        if source == 'agent':
            role = 'AGENT'
        elif source == 'user':
            role = 'USER'
        elif source == 'environment':
            role = 'ENV'
        else:
            role = _normalize_text(source).upper()

        # First, detect non-native embedded tool-call format in message text
        if _is_embedded_tool_call_message(message):
            parsed = _parse_embedded_tool_call(message)
            direction = '→'
            kind = f"action:{parsed['tool']}"
            content = parsed['content']

        # Then detect plain-text tool results from non-native format
        elif _is_tool_result_message(message):
            direction = '←'
            kind = 'obs:tool_result'
            content = message

        else:
            # Native structured format behavior (existing logic)
            if action:
                direction = '→'
                kind = f'action:{action}'
            elif observation:
                direction = '←'
                kind = f'obs:{observation}'
            else:
                direction = ' '
                kind = 'message'

            # Pick the most informative content per action/observation type
            if action == 'message':
                content = _normalize_text(args.get('content', '')) or message
            elif action == 'think':
                content = _normalize_text(args.get('thought', '')) or message or native_reasoning
            elif action == 'run':
                content = _normalize_text(args.get('command', '')) or message
            elif action in ('read', 'write', 'edit'):
                # args.command can be missing in native traces; fall back to action.
                cmd = _normalize_text(args.get('command', ''))
                label = cmd or action
                path = _normalize_text(args.get('path', ''))
                old_str = _normalize_text(args.get('old_str', ''))
                new_str = _normalize_text(args.get('new_str', ''))
                if cmd == 'str_replace' and (old_str or new_str):
                    content = f'[{label}] {path}\n--- old ---\n{old_str}\n+++ new +++\n{new_str}'
                elif cmd in ('view', 'create', 'insert', 'undo_edit'):
                    body = _normalize_text(args.get('file_text', args.get('new_str', message)))
                    content = f'[{label}] {path}' + (f'\n{body}' if body else '')
                else:
                    content = f'[{label}] {path}' if (label or path) else message
            elif observation:
                content = _normalize_text(entry.get('content', '')) or message
            else:
                content = _normalize_text(entry.get('content', '')) or _normalize_text(args.get('content', '')) or message

            # For native traces, many tool actions keep reasoning in model_response.
            if role == 'AGENT' and action in ('run', 'read', 'write', 'edit', 'browse', 'search', 'message'):
                content = _merge_reasoning(content, native_reasoning)

        # Truncate long content for readability
        if content and len(content) > max_message_len and role in ('ENV', 'AGENT'):
            content = content[:max_message_len] + f'... [truncated, total {len(content)} chars]'

        lines.append(f'[{i:3d}] {role:<5} {direction} {kind}: {content}')

    return '\n\n'.join(lines)


print(format_history_for_llm(instances[0]['history']))

[  0] AGENT → action:system: You are OpenHands agent, a helpful AI assistant that can interact with a computer to solve tasks.

<ROLE>
Your primary role is to assist users by executing commands, modifying code, and solving technical problems effectively. You should be thorough, methodical, and prioritize quality over speed.
* If the user asks a question, like "why is X happening", don't try to fix the problem. Just give an answer to the question.
</ROLE>

<EFFICIENCY>
* Each action you take is somewhat expensive. Wherever possible, combine multiple actions into a single action, e.g. combine multiple bash commands into one, using sed and grep to edit/view multiple files at once.
* When exploring the codebase, use efficient tools like find, grep, and git commands with appropriate filters to minimize unnecessary operations.
</EFFICIENCY>

<FILE_SYSTEM_GUIDELINES>
* When a user provides a file path, do NOT assume it's relative to the current working directory. First explore the file system

In [11]:

# ── Shared context builder ────────────────────────────────────────────────────

def _extract_context(inst: dict, max_traj_len: int = 200000) -> dict:
    """Extract all reusable context fields from an instance."""
    d = inst['instance']
    trajectory = format_history_for_llm(inst['history'], max_message_len=800)
    if len(trajectory) > max_traj_len:
        trajectory = trajectory[:max_traj_len] + f'\n\n... [trajectory truncated, total {len(trajectory)} chars]'
    return dict(
        problem=d.get('problem_statement', '').strip(),
        golden_patch=d.get('patch', '').strip(),
        test_patch=d.get('test_patch', '').strip(),
        generated_patch=inst.get('git_patch', '').strip() or '(no patch generated)',
        trajectory=trajectory,
    )


# ── Dimension-specific prompt builders ───────────────────────────────────────

def build_fix_quality_prompt(ctx: dict) -> tuple[str, str]:
    """Does the generated patch correctly fix the issue?"""
    system = """\
You are an expert software engineer. Evaluate whether the agent's generated patch
correctly fixes the reported GitHub issue, compared to the ground-truth patch.

The fix does NOT need to be identical — semantic equivalence is sufficient.
Focus only on correctness and completeness of the fix, not on style or tests.

Respond with a JSON object exactly matching this schema (no extra text):
{
  "reasoning": "<concise explanation>",
  "score": <int 0-4>
}

Scoring rubric:
  4 = Fully correct fix, matches intent of ground-truth patch
  3 = Mostly correct, minor gaps or slightly different approach
  2 = Partially correct, addresses some but not all aspects of the issue
  1 = Barely correct, largely misses the problem
  0 = Completely wrong or no patch generated"""

    user = f"""\
## Problem Statement
{ctx['problem']}

---
## Ground-Truth Patch (correct fix)
```diff
{ctx['golden_patch']}
```

---
## Agent-Generated Patch
```diff
{ctx['generated_patch']}
```
"""
    return system, user


def build_test_intent_prompt(ctx: dict) -> tuple[str, str]:
    """Does the fix address what the ground-truth tests verify?"""
    system = """\
You are an expert software engineer. Given a ground-truth test patch and an
agent-generated code patch, evaluate whether the agent's fix would satisfy the
intent of the ground-truth tests — i.e., would the ground-truth tests pass if
applied on top of the agent's patch?

Focus only on test coverage intent, not on patch style or unrelated behavior.

Respond with a JSON object exactly matching this schema (no extra text):
{
  "score": <int 0-4>,
  "reasoning": "<concise explanation>"
}

Scoring rubric:
  4 = Agent's fix fully satisfies what the tests verify
  3 = Mostly satisfies, minor gaps
  2 = Partially satisfies, some tests would likely fail
  1 = Barely satisfies, most tests would fail
  0 = Completely fails or no patch generated"""

    user = f"""\
## Problem Statement
{ctx['problem']}

---
## Ground-Truth Test Patch (what correctness looks like)
```diff
{ctx['test_patch']}
```

---
## Agent-Generated Patch
```diff
{ctx['generated_patch']}
```
"""
    return system, user


def build_trajectory_coherence_prompt(ctx: dict) -> tuple[str, str]:
    """Is the agent's reasoning and decision-making sound throughout?"""
    system = """\
You are an expert software engineer reviewing a code agent's execution trajectory
on a GitHub issue resolution task.

Evaluate the quality of the agent's reasoning and decision-making process.
Do NOT evaluate the final patch quality — focus solely on the trajectory itself.

Criteria to assess:
- Correct problem diagnosis: did the agent understand the issue?
- Logical planning: did it form a sensible plan before acting?
- Efficient tool use: no unnecessary retries, redundant searches, or wasted steps
- Consistent reasoning: no contradictions or abrupt direction changes
- Sound decisions: each step follows logically from the previous

Respond with a JSON object exactly matching this schema (no extra text):
{
  "score": <int 0-4>,
  "reasoning": "<concise explanation, highlight specific good/bad decisions>"
}

Scoring rubric:
  4 = Excellent reasoning throughout, efficient and consistent
  3 = Good overall, minor inefficiencies or one poor decision
  2 = Partial, some correct steps but notable confusion or wasted effort
  1 = Poor, mostly confused or thrashing with little forward progress
  0 = Completely incoherent or no trajectory"""

    user = f"""\
## Problem Statement
{ctx['problem']}

---
## Agent Trajectory
{ctx['trajectory']}
"""
    return system, user


# ── Registry: name → prompt builder ──────────────────────────────────────────
DIMENSION_BUILDERS = {
    "fix_quality":          build_fix_quality_prompt,
    "test_intent_coverage": build_test_intent_prompt,
    "trajectory_coherence": build_trajectory_coherence_prompt,
}


def print_dimension_prompt(inst: dict, dimension: str, **ctx_kwargs) -> None:
    """Print the prompt for a specific evaluation dimension without calling the LLM.

    Args:
        inst:      Instance dict.
        dimension: One of 'fix_quality', 'test_intent_coverage', 'trajectory_coherence'.
    """
    assert dimension in DIMENSION_BUILDERS, f"Unknown dimension: {dimension!r}. Choose from {list(DIMENSION_BUILDERS)}"
    ctx = _extract_context(inst, **ctx_kwargs)
    system, user = DIMENSION_BUILDERS[dimension](ctx)
    sep = "─" * 80
    print(f"{sep}\n[SYSTEM — {dimension}]\n{sep}")
    print(system)
    print(f"\n{sep}\n[USER — {dimension}]\n{sep}")
    print(user)


In [12]:

import os
import re
import json
from openai import OpenAI


def call_llm_judge(
    system_prompt: str,
    user_prompt: str,
    model: str = "Qwen/Qwen3-Coder-30B-A3B-Instruct",
    api_key: str = "not-needed",
    base_url: str = "http://localhost:8000/v1",
    extra_body: dict | None = None,
) -> dict:
    """Call an OpenAI-compatible LLM and return a parsed JSON dict."""
    client = OpenAI(api_key=api_key, base_url=base_url)
    kwargs = dict(
        model=model,
        temperature=0.0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        response_format={"type": "json_object"},
    )
    if extra_body:
        kwargs["extra_body"] = extra_body

    response = client.chat.completions.create(**kwargs)
    raw = response.choices[0].message.content

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass
    # Fallback: strip markdown fences and extract first {...} block
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if match:
        return json.loads(match.group())
    raise ValueError(f"Could not parse JSON from LLM response:\n{raw}")


def evaluate_instance(
    inst: dict,
    dimensions: list[str] | None = None,
    print_prompt: bool = False,
    **llm_kwargs,
) -> dict:
    """Run per-dimension LLM-as-judge evaluation on a single instance.

    Each dimension is evaluated with a separate, focused LLM call.

    Args:
        inst:        Instance dict from load_all_instances.
        dimensions:  Subset of dimensions to evaluate. Defaults to all three.
                     Choices: 'fix_quality', 'test_intent_coverage', 'trajectory_coherence'
        print_prompt: If True, print each dimension's prompt before calling the LLM.
        **llm_kwargs: Forwarded to call_llm_judge.

    Returns:
        Dict with per-dimension results and an averaged overall_score.
    """
    if dimensions is None:
        dimensions = list(DIMENSION_BUILDERS.keys())

    print(f"Evaluating [{inst['instance_id']}]  dimensions: {dimensions}")
    ctx = _extract_context(inst)
    results = {"instance_id": inst["instance_id"]}
    scores = []

    for dim in dimensions:
        assert dim in DIMENSION_BUILDERS, f"Unknown dimension: {dim!r}"
        system, user = DIMENSION_BUILDERS[dim](ctx)

        if print_prompt:
            sep = "─" * 80
            print(f"\n{sep}\n[PROMPT — {dim}]\n{sep}")
            print(f"[SYSTEM]\n{system}\n\n[USER]\n{user}")

        print(f"  · {dim} ...", end=" ", flush=True)
        dim_result = call_llm_judge(system, user, **llm_kwargs)
        results[dim] = dim_result
        scores.append(dim_result.get("score", 0))
        print(f"[{dim_result.get('score', '?')}/4]")

    results["overall_score"] = round(sum(scores) / len(scores), 2) if scores else 0
    return results


In [13]:

# ── LLM config — locally served vLLM endpoint ────────────────────────────────
LLM_CONFIG = dict(
    # model="Qwen/Qwen3-Coder-480B-A35B-Instruct",
    model="zai-org/GLM-5-FP8",
    api_key="not-needed",
    base_url="http://localhost:8000/v1",
    extra_body={"chat_template_kwargs": {"enable_thinking": True}},
)

# inst = next(i for i in instances if i["instance_id"] == "getmoto__moto-6784")
inst = instances[1]
# ── Inspect a single dimension's prompt (no LLM call) ────────────────────────
# print_dimension_prompt(inst, "fix_quality")
# print_dimension_prompt(inst, "test_intent_coverage")
# print_dimension_prompt(inst, "trajectory_coherence")

# ── Evaluate all dimensions (separate LLM call per dimension) ────────────────
result = evaluate_instance(inst, print_prompt=False, **LLM_CONFIG)

# ── Evaluate a subset of dimensions ──────────────────────────────────────────
# result = evaluate_instance(inst, dimensions=["fix_quality", "test_intent_coverage"], **LLM_CONFIG)

# ── Pretty-print results ──────────────────────────────────────────────────────
print()
for dim in ("fix_quality", "test_intent_coverage", "trajectory_coherence"):
    if dim not in result:
        continue
    score = result[dim]["score"]
    reason = result[dim]["reasoning"]
    print(f"{dim:28s}  [{score}/4]  {reason}")

print(f"\nOverall score (avg): {result['overall_score']}/4")


Evaluating [getmoto__moto-5876]  dimensions: ['fix_quality', 'test_intent_coverage', 'trajectory_coherence']
  · fix_quality ... [2/4]
  · test_intent_coverage ... [3/4]
  · trajectory_coherence ... [4/4]

fix_quality                   [2/4]  The agent's patch correctly identifies the need for an AliasExistsException and adds validation to admin_update_user_attributes. However, it has several gaps compared to the ground-truth: (1) It doesn't handle the update_user_attributes method which also needs this validation per the ground-truth. (2) The approach uses user_pool._get_user(attr_value) which may not correctly find users by attribute value - the ground-truth iterates through users and checks attribute_lookup which is more reliable. (3) The agent's approach is more general (checks all UsernameAttributes) which could be good, but the lookup mechanism is questionable. The core functionality is partially implemented but with notable gaps.
test_intent_coverage          [3/4]  The agent's 

In [20]:

# Inspect any single dimension's prompt
print_dimension_prompt(inst, "fix_quality")
# print_dimension_prompt(inst, "test_intent_coverage")
# print_dimension_prompt(inst, "trajectory_coherence")


────────────────────────────────────────────────────────────────────────────────
[SYSTEM — fix_quality]
────────────────────────────────────────────────────────────────────────────────
You are an expert software engineer. Evaluate whether the agent's generated patch
correctly fixes the reported GitHub issue, compared to the ground-truth patch.

The fix does NOT need to be identical — semantic equivalence is sufficient.
Focus only on correctness and completeness of the fix, not on style or tests.

Respond with a JSON object exactly matching this schema (no extra text):
{
  "reasoning": "<concise explanation>",
  "score": <int 0-4>
}

Scoring rubric:
  4 = Fully correct fix, matches intent of ground-truth patch
  3 = Mostly correct, minor gaps or slightly different approach
  2 = Partially correct, addresses some but not all aspects of the issue
  1 = Barely correct, largely misses the problem
  0 = Completely wrong or no patch generated

─────────────────────────────────────────────────

In [17]:
print(inst.keys())

dict_keys(['instance_id', 'instance', 'history', 'git_patch', 'validation_log', 'validation_map', 'metadata', 'instruction', 'metrics', 'error'])


In [85]:

import pathlib
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm


def _evaluate_one(inst: dict, dims: list[str], llm_config: dict) -> dict:
    """Evaluate a single instance across all requested dimensions. Thread-safe."""
    ctx = _extract_context(inst)
    scores = []
    judgment = {"instance_id": inst["instance_id"]}
    for dim in dims:
        system, user = DIMENSION_BUILDERS[dim](ctx)
        dim_result = call_llm_judge(system, user, **llm_config)
        judgment[dim] = dim_result
        scores.append(dim_result.get("score", 0))
    judgment["overall_score"] = round(sum(scores) / len(scores), 2) if scores else 0
    return judgment


def run_batch_evaluation(
    instances: list[dict],
    llm_config: dict,
    max_instances: int | None = 10,
    output_path: str = "judge_results.jsonl",
    dimensions: list[str] | None = None,
    skip_existing: bool = True,
    max_workers: int = 4,
) -> list[dict]:
    """Evaluate up to max_instances instances in parallel and save results to JSONL.

    Each instance is evaluated with one LLM call per dimension. Workers run
    concurrently via a thread pool (safe for I/O-bound HTTP calls to vLLM).
    Results are written atomically as each future completes; safe to re-run
    with skip_existing=True to resume after interruption.

    Args:
        instances:      Full list of instances from load_all_instances.
        llm_config:     LLM kwargs forwarded to call_llm_judge.
        max_instances:  Cap on how many new instances to evaluate. None = all.
        output_path:    Path to the JSONL file where results are appended.
        dimensions:     Dimension names to evaluate. None = all three.
        skip_existing:  If True, skip instance_ids already in the output file.
        max_workers:    Number of parallel threads (tune to vLLM concurrency).
    """
    _dims = dimensions if dimensions is not None else list(DIMENSION_BUILDERS.keys())
    out = pathlib.Path(output_path)
    write_lock = threading.Lock()

    # Load already-completed instance_ids so we can resume
    completed_ids: set[str] = set()
    if skip_existing and out.exists():
        with out.open() as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        completed_ids.add(json.loads(line)["instance_id"])
                    except Exception:
                        pass
        if completed_ids:
            print(f"Skipping {len(completed_ids)} already-evaluated instance(s).")

    todo = [i for i in instances if i["instance_id"] not in completed_ids]
    if max_instances is not None:
        todo = todo[:max_instances]

    print(f"Will evaluate {len(todo)} instance(s) with {max_workers} workers → {out}")
    new_results = []

    with out.open("a") as f, ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(_evaluate_one, inst, _dims, llm_config): inst
            for inst in todo
        }

        with tqdm(total=len(futures), desc="Evaluating", unit="inst") as pbar:
            for future in as_completed(futures):
                inst = futures[future]
                iid = inst["instance_id"]
                try:
                    judgment = future.result()
                    scores_str = "  ".join(
                        f"{d}={judgment[d]['score']}/4"
                        for d in _dims if d in judgment
                    )
                    pbar.write(f"✓ {iid}  {scores_str}  overall={judgment['overall_score']}")
                except Exception as e:
                    judgment = {"instance_id": iid, "error": str(e)}
                    pbar.write(f"✗ {iid}  ERROR: {e}")

                with write_lock:
                    f.write(json.dumps(judgment) + "\n")
                    f.flush()

                new_results.append(judgment)
                pbar.update(1)

    print(f"\nDone. {len(new_results)} new result(s) saved to {out}")
    return new_results


In [87]:

# ── Batch evaluation config ───────────────────────────────────────────────────
BATCH_OUTPUT  = "judge_results.jsonl"
MAX_INSTANCES = 500    # set to None to evaluate all
MAX_WORKERS   = 16      # parallel threads — tune to your vLLM server concurrency

batch_results = run_batch_evaluation(
    instances=instances,
    llm_config=LLM_CONFIG,
    max_instances=MAX_INSTANCES,
    output_path=BATCH_OUTPUT,
    dimensions=None,       # None = all three dimensions
    skip_existing=True,    # resume from where you left off
    max_workers=MAX_WORKERS,
)

# ── Summary stats ─────────────────────────────────────────────────────────────
valid = [r for r in batch_results if "error" not in r]
errors = [r for r in batch_results if "error" in r]
print(f"\n{len(valid)} succeeded, {len(errors)} failed")
if valid:
    for dim in ("fix_quality", "test_intent_coverage", "trajectory_coherence"):
        scores = [r[dim]["score"] for r in valid if dim in r]
        if scores:
            print(f"{dim:28s}  avg={sum(scores)/len(scores):.2f}  n={len(scores)}")
    overall = [r["overall_score"] for r in valid]
    print(f"{'overall_score':28s}  avg={sum(overall)/len(overall):.2f}")


Skipping 200 already-evaluated instance(s).
Will evaluate 500 instance(s) with 16 workers → judge_results.jsonl


Evaluating:   0%|          | 0/500 [00:00<?, ?inst/s]

✓ getmoto__moto-7434  fix_quality=4/4  test_intent_coverage=4/4  trajectory_coherence=4/4  overall=4.0
✓ getmoto__moto-5439  fix_quality=3/4  test_intent_coverage=2/4  trajectory_coherence=4/4  overall=3.0
✓ getmoto__moto-7647  fix_quality=3/4  test_intent_coverage=4/4  trajectory_coherence=4/4  overall=3.67
✓ getmoto__moto-7439  fix_quality=4/4  test_intent_coverage=4/4  trajectory_coherence=4/4  overall=4.0
✓ getmoto__moto-5919  fix_quality=0/4  test_intent_coverage=4/4  trajectory_coherence=4/4  overall=2.67
✓ getmoto__moto-5980  fix_quality=3/4  test_intent_coverage=4/4  trajectory_coherence=4/4  overall=3.67
✓ getmoto__moto-6716  fix_quality=2/4  test_intent_coverage=4/4  trajectory_coherence=4/4  overall=3.33
✓ getmoto__moto-5109  fix_quality=4/4  test_intent_coverage=4/4  trajectory_coherence=4/4  overall=4.0
✓ getmoto__moto-5880  fix_quality=3/4  test_intent_coverage=3/4  trajectory_coherence=4/4  overall=3.33
✓ getmoto__moto-5444  fix_quality=2/4  test_intent_coverage=3/4  tra